# Первая практика — мини-анализ HMDA

## Подготовка

In [1]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: C:\Users\ASUS\data-analysis-course\.venv\Scripts\python.exe
pandas: 2.2.2
Рабочая папка: C:\Users\ASUS\data-analysis-course\lessons\01-introduction\work


## Маленький пример reading_example.csv

In [2]:
example = pd.read_csv(
    DATA / "reading_example.csv",
    dtype={"lei": "string", "county_code": "string"},
)
example

,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,275000.0,72.0
1,3,2023,549300P1347BFCBHJV60,11001,1,585000.0,174.0
2,278,2023,KB1H1DSPRFMYMCUFXT09,11001,2,475000.0,108.0
3,304,2023,549300VZVN841I2ILS84,11001,2,905000.0,353.0
4,21,2023,549300RPOGWJRH63HS39,11001,3,595000.0,184.0
5,27,2023,JJKC32MCHWDI71265Z06,11001,3,1805000.0,667.0


## A. Разделитель CSV

In [3]:
wrong_sep = pd.read_csv(
    DATA / "reading_example.csv",
    sep=";"
)

wrong_sep

,"source_row,activity_year,lei,county_code,action_taken,loan_amount,income"
0,"2,2023,549300VZVN841I2ILS84,11001,1,275000.0,72.0"
1,"3,2023,549300P1347BFCBHJV60,11001,1,585000.0,1..."
2,"278,2023,KB1H1DSPRFMYMCUFXT09,11001,2,475000.0..."
3,"304,2023,549300VZVN841I2ILS84,11001,2,905000.0..."
4,"21,2023,549300RPOGWJRH63HS39,11001,3,595000.0,..."
5,"27,2023,JJKC32MCHWDI71265Z06,11001,3,1805000.0..."


In [4]:
wrong_sep = pd.read_csv(
    DATA / "reading_example.csv",
    sep=";"
)

wrong_sep.shape

(6, 1)

## B. Частичное чтение

In [5]:
preview = pd.read_csv(
    DATA / "reading_example.csv",
    nrows=3
)

preview

small_columns = pd.read_csv(
    DATA / "reading_example.csv",
    usecols=["action_taken", "income"]
)

small_columns

,action_taken,income
0,1,72.0
1,1,174.0
2,2,108.0
3,2,353.0
4,3,184.0
5,3,667.0


In [6]:
print("preview:", preview.shape)
print("small_columns:", small_columns.shape)

preview: (3, 7)
small_columns: (6, 2)


## Выбор Series и DataFrame

In [7]:
outcomes = example["action_taken"]
print(type(outcomes))
outcomes

<class 'pandas.core.series.Series'>


0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: int64

## Выбор нескольких столбцов

In [8]:
names = ["source_row", "action_taken", "income"]
selected = example[names]
print(type(selected))
selected.head(6)

<class 'pandas.core.frame.DataFrame'>


,source_row,action_taken,income
0,2,1,72.0
1,3,1,174.0
2,278,2,108.0
3,304,2,353.0
4,21,3,184.0
5,27,3,667.0


## C. Выбор столбцов и просмотр

In [9]:
amount_view = example[["source_row", "loan_amount"]]

amount_view.head(2)

,source_row,loan_amount
0,2,275000.0
1,3,585000.0


In [10]:
amount_view.shape

(6, 2)

In [11]:
amount_view.columns

Index(['source_row', 'loan_amount'], dtype='object')

In [12]:
type(example)

pandas.core.frame.DataFrame

## Пример расчёта одобрений

In [13]:
approved_flag = outcomes.isin([1, 2])
approved_flag

0     True
1     True
2     True
3     True
4    False
5    False
Name: action_taken, dtype: bool

## Количество и доля одобрений

In [14]:
n_approved = approved_flag.sum()
n_decisions = len(example)
assert n_decisions > 0
small_approval_rate = n_approved / n_decisions
print(n_approved, n_decisions)
print(f"Одобрено: {100 * small_approval_rate:.2f}%")

4 6
Одобрено: 66.67%


## D. Доля выдач на маленьком примере

In [15]:
approved_flag = outcomes.isin([1, 2])

n_approved = approved_flag.sum()
n_decisions = len(example)

small_approval_rate = n_approved / n_decisions
issued_flag = outcomes == 1

issued_flag

0     True
1     True
2    False
3    False
4    False
5    False
Name: action_taken, dtype: bool

In [16]:
n_issued = issued_flag.sum()
n_decisions = len(example)

small_origination_rate = n_issued / n_decisions

print(n_issued, n_decisions)
print(f"Выдано: {100 * small_origination_rate:.2f}%")

2 6
Выдано: 33.33%


## Основной набор decisions.csv

In [17]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
full_outcomes = decisions["action_taken"]
full_n = len(decisions)
print(decisions.shape)
full_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## E. Две доли на основном наборе

In [18]:
full_approved_mask = full_outcomes.isin([1, 2])
full_approved = full_approved_mask.sum()

full_issued_mask = full_outcomes == 1
full_issued = full_issued_mask.sum()

full_approval_rate = full_approved / full_n
full_origination_rate = full_issued / full_n

print(f"Всего записей: {full_n}")
print(f"Одобрено (1 и 2): {full_approved}, Доля одобрений: {full_approval_rate:.2%}")
print(f"Выдано (1): {full_issued}, Доля выдач: {full_origination_rate:.2%}")

print("\nПример первых 5 значений full_approved_mask:", full_approved_mask.head().tolist())

Всего записей: 6830
Одобрено (1 и 2): 6139, Доля одобрений: 89.88%
Выдано (1): 5956, Доля выдач: 87.20%

Пример первых 5 значений full_approved_mask: [True, True, True, True, True]


## Проверка входных данных

In [19]:
assert full_n > 0
assert full_outcomes.isna().sum() == 0
assert full_outcomes.isin([1, 2, 3]).sum() == full_n
income_missing = decisions["income"].isna().sum()
print("Пропущен доход:", income_missing)

Пропущен доход: 342


## F. Баланс и диапазоны

In [20]:
full_denied_mask = full_outcomes == 3
full_denied = full_denied_mask.sum()

assert full_approved + full_denied == full_n, "Сумма одобрений и отказов не равна общему числу записей!"

assert full_issued >= 0, "Количество выдач не может быть отрицательным!"
assert full_issued <= full_approved, "Выдач не может быть больше, чем одобрений!"
assert full_approved <= full_n, "Одобрений не может быть больше, чем всего записей!"

assert 0 <= full_approval_rate <= 1, "Доля одобрений выходит за пределы [0, 1]!"
assert 0 <= full_origination_rate <= 1, "Доля выдач выходит за пределы [0, 1]!"

print("Все проверки (assert) пройдены успешно!")
print(f"Отказов (3): {full_denied}")
print(f"Сумма: {full_approved} (одобрено) + {full_denied} (отказ) = {full_approved + full_denied} (всего: {full_n})")

Все проверки (assert) пройдены успешно!
Отказов (3): 691
Сумма: 6139 (одобрено) + 691 (отказ) = 6830 (всего: 6830)


## G. Расчёт из отчёта

In [21]:
approval_rate = full_outcomes.eq(1).sum() / full_n
approval_rate

0.8720351390922401

## G. Исправление условия

In [22]:
approval_rate_fixed = full_outcomes.isin([1, 2]).sum() / full_n
approval_rate_fixed

0.8988286969253294

## G. Исправление названия

In [23]:
origination_rate_fixed = full_outcomes.eq(1).sum() / full_n
origination_rate_fixed

0.8720351390922401